## Eni Mozambique Data
---
 This notebook is to read and extract recorded information from the `.txt` files provided, and write NetCDFs with the same file names.
 
 `version 1.0, May, 8, 2026, auth:moeindst`
 
 ---

In [ ]:
### importing libraries and packages
from pathlib import Path
import re
from io import StringIO

import xarray as xr
import pandas as pd
import os

---
### Clustering to speed up procedure
Info on clustering with control on the number of workers and threads:

* 4 workers × 2 threads = 8 Dask threads

    4 workers × 8 GB = 32 GB Dask memory limit


* `free -h` for quick info on machine's memory and `lscpu` for cpu and threads information.

In [ ]:
cluster_type = 'Local'
#cluster_type = 'Coiled'
#cluster_type = 'Gateway'

In [ ]:
if cluster_type == 'Local':
    
    from dask.distributed import LocalCluster, Client

    cluster = LocalCluster(
        n_workers=4,
        threads_per_worker=2,
        memory_limit="8GB",
        processes=True,
        dashboard_address=":8181"
                            )

    client = Client(cluster)
    
### 4 workers × 2 threads = 8 Dask threads
### 4 workers × 8 GB = 32 GB Dask memory limit

In [ ]:
client

---

### Reading the text files to create NetCDF versions
* There are different parsers based on the instruments and file names that deal with the various ways the data had been written. These parsers read the header parts and put them as attributes/metadata into the netcdf file.

In [ ]:
txt_path = '~/Data/EniMozambique_extremeWaveStorm/data/txt/'

In [ ]:
import eni_data_read_funcs as func

In [ ]:
%%time

txt_path = Path("../data/txt")
out_path = Path("../data/netcdf")

failed_files = []
skipped_files = []

for file_path in sorted(txt_path.glob("*.txt")):
    name = file_path.name.upper()

    try:
        if "SPECTRA" in name:
            print(f"SKIPPING spectra: {file_path.name}")
            skipped_files.append(file_path.name)
            continue

        elif "_MET" in name or "WAVESCAN_MET" in name:
            ds = func.parse_met_txt_to_dataset(file_path)

        elif "_WAVES" in name or "_WAVE.TXT" in name:
            ds = func.parse_wave_txt_to_dataset(file_path)

        elif "_WLR" in name:
            ds = func.parse_wlr_txt_to_dataset(file_path)

        elif "_RBR" in name:
            ds = func.parse_rbr_txt_to_dataset(file_path)

        elif "_SBE37" in name:
            ds = func.parse_sbe37_txt_to_dataset(file_path)

        elif "_SG_" in name:
            ds = func.parse_sg_txt_to_dataset(file_path)

        elif (
            "WHADCP" in name
            or "WHWADCP" in name
            or "AQUADOPP_CURRENT" in name
            or "WHW_CURRENT" in name
            or "LRADCP" in name
        ):
            ds = func.parse_profile_current_txt_to_dataset(file_path)

        else:
            print(f"SKIPPING unknown type: {file_path.name}")
            skipped_files.append(file_path.name)
            continue

        # common post-processing
        ds = func.chunk_dataset_adaptive(ds, target_time_chunk=1000)

        out_file = out_path / f"{file_path.stem}.nc"

        ds.to_netcdf(
            out_file,
            engine="netcdf4",
            format="NETCDF4"
        )

        print(f"WRITTEN: {out_file.name}")

    except Exception as e:
        print(f"FAILED: {file_path.name} -> {e}")
        failed_files.append((file_path.name, str(e)))

In [ ]:
out_path = Path("../data/netcdf")
f2open = 'Final_C1_SBE37_Upper.nc'

ds_inspect = xr.open_dataset(os.path.join(out_path,f2open))
ds_inspect

                              <------------------ End of Document ------------------>